# 13 — Proof-of-concept story and visual summary

## Heat-conditioned accessibility in Sevilla

This notebook is the **entry point to the PoC repository**. It loads saved outputs from the preceding analytical notebooks and tells the methodological story without recomputing the full pipeline.

The central research idea is:

**urban form → thermal environment → thermal network impedance → accessibility loss → observed mobility response**

The current thermal simulation covers only a small proof-of-concept domain within Sevilla. Results should therefore be interpreted as methodological demonstrations, not citywide impact estimates.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import rasterio

from matplotlib.patches import FancyBboxPatch
from shapely.geometry import box

ACCESS_DIR = OUTPUTS / "accessibility"
SOLWEIG_DIR = OUTPUTS / "solweig" / "summary"
MOBILE_PROCESSED = PROJECT_ROOT / "data" / "mobile" / "processed"

## 1. Scientific workflow

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))
ax.set_xlim(0, 15)
ax.set_ylim(0, 5)
ax.axis("off")


def workflow_box(x, y, width, height, title, subtitle):
    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.04,rounding_size=0.12",
        linewidth=1.2,
        facecolor="white",
        edgecolor="0.25",
    )
    ax.add_patch(patch)

    ax.text(
        x + width / 2,
        y + height * 0.65,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
    )

    ax.text(
        x + width / 2,
        y + height * 0.32,
        subtitle,
        ha="center",
        va="center",
        fontsize=9,
        color="0.35",
    )


def arrow(x1, y1, x2, y2):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="-|>",
            lw=1.4,
            color="0.35",
        ),
    )


y, h, w = 2.55, 1.25, 2.15

workflow_box(0.25, y, w, h, "Urban form", "Buildings\nVegetation\nStreet network")
workflow_box(3.05, y, w, h, "Thermal environment", "SOLWEIG\nUTCI / Tmrt")
workflow_box(5.85, y, w, h, "Thermal network", "Thermal conditions\nassigned to edges")
workflow_box(8.65, y, w, h, "Generalized cost", "Walking time +\nthermal penalty")
workflow_box(11.45, y, w, h, "Accessibility", "Reachable network\nTransit opportunities")

arrow(2.40, 3.18, 3.05, 3.18)
arrow(5.20, 3.18, 5.85, 3.18)
arrow(8.00, 3.18, 8.65, 3.18)
arrow(10.80, 3.18, 11.45, 3.18)

workflow_box(8.65, 0.45, 2.15, 1.05, "Observed mobility", "Mobile-phone OD\nHot vs reference days")
workflow_box(11.45, 0.45, 2.15, 1.05, "Future comparison", "Modeled accessibility\nvs observed response")

arrow(10.80, 0.98, 11.45, 0.98)

ax.annotate(
    "",
    xy=(12.52, 1.50),
    xytext=(12.52, 2.55),
    arrowprops=dict(
        arrowstyle="-|>",
        lw=1.3,
        color="0.35",
    ),
)

ax.text(
    7.0,
    4.55,
    "Heat-conditioned accessibility proof-of-concept",
    ha="center",
    fontsize=17,
    fontweight="bold",
)

ax.text(
    7.0,
    4.20,
    "From urban form to thermal impedance, accessibility and observed mobility",
    ha="center",
    fontsize=10,
    color="0.4",
)

plt.tight_layout()
plt.show()

## 2. Core methodological idea

Conventional walking accessibility typically uses travel time or distance as impedance. The PoC adds a thermal penalty directly to each walking edge:

\[
c_e^{heat}=t_e[1+\lambda\max(UTCI_e-32,0)]
\]

where \(t_e\) is conventional walking time and \(\lambda\) controls sensitivity to thermal stress.

The output is a **generalized walking cost**, not normalized UTCI.

## 3. Study area, urban form, and UTCI footprint

In [ ]:
building_path = DATA_INTERIM / "sevilla_buildings_voxcity.gpkg"
tree_path = DATA_INTERIM / "sevilla_trees_voxcity.gpkg"
graph_path = DATA_RAW / "sevilla_walk.graphml"
utci_path = SOLWEIG_DIR / "utci_max.tif"

G0 = ox.load_graphml(graph_path)
_, edges0 = ox.graph_to_gdfs(G0)
target_crs = edges0.crs

fig, ax = plt.subplots(figsize=(9, 9))

edges0.plot(
    ax=ax,
    linewidth=0.45,
    alpha=0.40,
)

if building_path.exists():
    buildings = gpd.read_file(building_path).to_crs(target_crs)
    buildings.plot(
        ax=ax,
        alpha=0.18,
        edgecolor="0.35",
        linewidth=0.15,
    )

if tree_path.exists():
    trees = gpd.read_file(tree_path).to_crs(target_crs)
    tree_points = trees[trees.geometry.geom_type == "Point"]

    if len(tree_points):
        tree_points.plot(
            ax=ax,
            markersize=5,
            alpha=0.50,
        )

if utci_path.exists():
    with rasterio.open(utci_path) as src:
        utci_extent = gpd.GeoDataFrame(
            geometry=[
                box(
                    src.bounds.left,
                    src.bounds.bottom,
                    src.bounds.right,
                    src.bounds.top,
                )
            ],
            crs=src.crs,
        ).to_crs(target_crs)

    utci_extent.boundary.plot(
        ax=ax,
        linewidth=2.2,
        linestyle="--",
        zorder=10,
    )

ax.set_title("Sevilla walking network and thermal PoC domain")
ax.text(
    0.02,
    0.02,
    "Dashed boundary = SOLWEIG / UTCI simulation extent",
    transform=ax.transAxes,
    fontsize=9,
)
ax.set_axis_off()

plt.tight_layout()
plt.show()

**Interpretation.** The broader walking network provides spatial context, but the thermalized accessibility result represents only the limited SOLWEIG domain. It should not be read as a Sevilla-wide accessibility estimate.

## 4. Thermal simulation

In [ ]:
if utci_path.exists():
    with rasterio.open(utci_path) as src:
        utci = src.read(1, masked=True)
        extent = [
            src.bounds.left,
            src.bounds.right,
            src.bounds.bottom,
            src.bounds.top,
        ]

    fig, ax = plt.subplots(figsize=(8, 7))

    image = ax.imshow(
        utci,
        extent=extent,
        origin="upper",
    )

    ax.set_title("Daily-maximum UTCI stress-test surface")
    ax.set_xlabel("Easting")
    ax.set_ylabel("Northing")
    plt.colorbar(image, ax=ax, label="UTCI (°C)")

    plt.tight_layout()
    plt.show()

**Interpretation.** The surface demonstrates a complete urban-form → microclimate pipeline. Its limitation is equally important: cell-wise daily maximum UTCI is severe over most of the small domain and therefore produces relatively homogeneous edge penalties.

## 5. Thermalized walking network

In [ ]:
thermal_graph = ox.load_graphml(
    DATA_INTERIM / "sevilla_walk_thermal.graphml"
)

_, thermal_edges = ox.graph_to_gdfs(thermal_graph)

thermal_edges["cost_ratio"] = pd.to_numeric(
    thermal_edges["cost_ratio"],
    errors="coerce",
)

fig, ax = plt.subplots(figsize=(8, 8))

thermal_edges.plot(
    ax=ax,
    column="cost_ratio",
    linewidth=1.3,
    legend=True,
)

ax.set_title("Heat-adjusted walking cost multiplier")
ax.set_axis_off()

plt.tight_layout()
plt.show()

display(thermal_edges["cost_ratio"].describe())

The `cost_ratio` is:

\[
c_e^{heat}/t_e
\]

A value of 1.48 means the edge carries a 48% higher generalized cost than conventional walking time. It is **not** normalized UTCI.

## 6. Accessibility sensitivity

In [ ]:
sensitivity = pd.read_csv(
    ACCESS_DIR / "accessibility_sensitivity.csv"
)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    sensitivity["lambda"],
    sensitivity["mean_access_loss_pct"],
    marker="o",
)

ax.set_xlabel("Thermal penalty parameter λ")
ax.set_ylabel("Mean accessibility loss (%)")
ax.set_title("Sensitivity of transit-stop accessibility")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

display(sensitivity)

## 7. Origin-level accessibility loss

In [ ]:
access = pd.read_csv(
    ACCESS_DIR / "origin_accessibility_lambda_005.csv"
)

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(
    access["normal_access"],
    access["thermal_access"],
    alpha=0.75,
)

limit = max(
    access["normal_access"].max(),
    access["thermal_access"].max(),
)

ax.plot([0, limit], [0, limit], linestyle="--")

ax.set_xlabel("Normal reachable transit stops")
ax.set_ylabel("Thermal reachable transit stops")
ax.set_title("Normal vs heat-conditioned accessibility")

plt.tight_layout()
plt.show()

display(
    access[
        ["normal_access", "thermal_access", "access_loss_pct"]
    ].describe()
)

**Early interpretation.** At \(\lambda=0.05\), the PoC produces a large modeled reduction in reachable transit stops. This magnitude is a stress-test result tied to the local thermal field and selected sensitivity parameter—not a citywide Sevilla estimate.

## 8. Observed hot/reference mobility response

In [ ]:
mobility_path = (
    MOBILE_PROCESSED
    / "sevilla_district_hot_reference_mobility.parquet"
)

if mobility_path.exists():
    mobility = pd.read_parquet(mobility_path)

    plot_df = mobility.sort_values("change_pct")

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.barh(
        plot_df["district_name"],
        plot_df["change_pct"],
    )

    ax.axvline(0, linewidth=1)
    ax.set_xlabel("Hot-day mobility change relative to reference days (%)")
    ax.set_ylabel("Origin district")
    ax.set_title("Observed district-level mobility change")

    plt.tight_layout()
    plt.show()

**Spatial-scale warning.** The mobile-phone analysis covers all 11 Sevilla districts, whereas the thermal PoC represents only a small local domain. The current modeled and observed effects therefore should not yet be treated as directly comparable citywide quantities.

## 9. What the PoC demonstrates

The cleaned workflow establishes that:

1. urban-form inputs can be prepared for microclimate simulation;
2. SOLWEIG can generate a spatial thermal field;
3. the thermal field can be transferred to a pedestrian network;
4. thermal stress can be embedded inside generalized walking cost;
5. thermal impedance can alter routes;
6. it can reduce reachable transit opportunities within a fixed cost budget;
7. accessibility changes monotonically with \(\lambda\);
8. the opportunity-level analysis reproduces the accessibility loss mechanism exactly;
9. the daily-max UTCI stress-test surface is too spatially saturated for strong urban-form interpretation;
10. mobile-phone OD data provide a promising independent validation layer once the thermal model is spatially expanded.

## 10. Next-stage experiment

The next version should move toward:

**urban form → common hot-afternoon UTCI → heterogeneous edge penalties → city-scale accessibility → district-level observed mobility response**

Common-hour surfaces such as 14:00, 15:00, or 16:00 are preferable to cell-wise daily maximum because they preserve spatial differences in shade, radiation and urban morphology at the same moment.